# Overall performance of module detection methods

In [1]:
%env NUMBA_NUM_THREADS=1
%env MKL_NUM_THREADS=1
%env OPEN_BLAS_NUM_THREADS=1
%env NUMEXPR_NUM_THREADS=1
%env OMP_NUM_THREADS=1

env: NUMBA_NUM_THREADS=1
env: MKL_NUM_THREADS=1
env: OPEN_BLAS_NUM_THREADS=1
env: NUMEXPR_NUM_THREADS=1
env: OMP_NUM_THREADS=1


In [2]:
import sys
import os
# sys.path.insert(0,os.path.abspath("../lib/"))

import json

from util import JSONExtendedEncoder

%load_ext autoreload
%autoreload 2

%matplotlib inline
from matplotlib.pyplot import *

import pandas as pd
import numpy as np

import multiprocessing as mp

from itertools import product

import itertools
import shutil

import os

conf_folder = "conf/"

# Settings

In [3]:
n_jobs = 1 # int(mp.cpu_count() - 1)
method_name = None

In [4]:
# Parameters
method_name = "clamp_base"
n_jobs = 10


In [5]:
assert method_name is not None, "You have to specify a method_name"

In [6]:
print(f"Using {n_jobs} cores")

Using 10 cores


# Running a method on different parameter settings and datasets

Note: If you downloaded the results from zenodo, you don't need to rerun this for "dummy", "agglom", "ica_zscore", "spectral_biclust" and "meanshift"

The following code will explore the parameters of a module detection method on every dataset using a grid-search approach.

If you want to run your own method, you should wrap it into a python function and add its parameters to `conf/paramexplo_blueprints.py`. We will show the whole workflow here for a "dummy"  (but fast) clustering method, which will simply group genes randomly.

Every module detection method is wrapped in a python function (see `scripts/moduledetection.py`)

Because module detection methods usually take a while to run, we generate the files necessary to run a method on the several parameter settings and datasets here. These can then be easily called from the commandline, for example on a computer cluster or locally using GNU `parallel`.

This function will be called by scripts/moduledetection.py , which will save the modules in the correct format along with additional run information (such as running times).

In [7]:
# datasets to run
datasetnames = [
    "ecoli_colombos",
    "ecoli_dream5",
    "yeast_gpl2529",
    "yeast_dream5",
    "synth_ecoli_regulondb",
    "synth_yeast_macisaac",
    "human_tcga",
    "human_gtex",
    "human_seek_gpl5175",
    "ecoli_precise2"
]

# choose the method to evaluate
# method_name = "agglom_pearson_abs" # use the dummy method to check if everything works correctly
# method_name = "agglom" # this method runs very fast, and has the best performance among clustering methods
# method_name = "ica_zscore" # this method runs very slow, but has approx. the highest performance in the benchmark
# method_name = "spectral_biclust" # top biclustering method
# method_name = "meanshift"

To add your own method, create a function with "your_method_name" in the `lib/clustering.py` file (or any other file as long as it's imported in `scripts/moduledetection.py`.
This function should accept an `E` object (which is a dataframe with genes in columns) and any additional parameters
Then add reasonable parameter setting of your method to `conf/paramexplo_blueprints.py`.

method_name = "your_method_name"

In [8]:
# paramexplo_blueprints.py stores for every method the parameters which will be varied using a grid-search approach.
%run ../conf/paramexplo_blueprints.py
methodblueprint = blueprints[method_name]

In [9]:
methodblueprint

{'staticparams': {'method': 'clamp_base'},
 'dynparams': {'k': array([ 25.,  50.,  75., 100., 125., 150., 175., 200., 225., 250., 275.,
         300.]),
  'qvalcutoff': array([1.e-01, 1.e-02, 1.e-03, 1.e-04, 1.e-05, 1.e-06, 1.e-07, 1.e-08,
         1.e-09, 1.e-10])},
 'type': 'moduledetection'}

Generate different parameter settings using a grid-search.

In [10]:
params_folder = "conf/paramexplo/" + method_name + "/"
if os.path.exists("../" + params_folder):
    shutil.rmtree("../" + params_folder)
os.makedirs("../" + params_folder)

methodsettings = []
method_locations = []
i = 0
for dynparam_combination in list(itertools.product(*[methodblueprint["dynparams"][param] for param in sorted(methodblueprint["dynparams"].keys())])):
    method = {"params":{}}
    method["params"] = methodblueprint["staticparams"].copy()
    method["params"].update(dict(zip(sorted(methodblueprint["dynparams"].keys()), dynparam_combination)))
    method["location"] = params_folder + str(i) + ".json"
    method["seed"] = 0

    methodsettings.append(method)

    json.dump(method, open("../" + method["location"], "w"), cls=JSONExtendedEncoder)

    method_locations.append(method["location"])

    i+=1

Now combine the different parameter settings and datasets. Then generate the different python commands to run every parameter setting and dataset in parallel.

In [11]:
settings_name = "paramexplo/{method_name}".format(method_name = method_name)
settings = []
for datasetname in datasetnames:
    for setting_ix, methodsetting in enumerate(methodsettings):
        settingid = datasetname + "_" + str(setting_ix)
        settings.append({
            "dataset_location":"conf/datasets/" + datasetname + ".json",
            "dataset_name":datasetname,
            "method_location":methodsetting["location"],
            "output_folder":"results/" + methodblueprint["type"] + "/{settings_name}/{settingid}/".format(settings_name=settings_name, settingid=settingid),
            "settingid":settingid
        })
json.dump(settings, open("../conf/settings/{settings_name}.json".format(settings_name=settings_name), "w"))

In [12]:
settings_dataset = pd.DataFrame([dict(settingid=setting["settingid"], **json.load(open("../" + setting["dataset_location"]))["params"]) for setting in settings])
settings_method = pd.DataFrame([dict(settingid=setting["settingid"], **json.load(open("../" + setting["method_location"]))["params"]) for setting in settings])

In [13]:
# commands = ""
# for i, setting in enumerate(settings):
#     #commands += "python scripts/moduledetection.py {method_location} {dataset_location} {output_folder} 0 test\n".format(**setting)
#     commands += "python3 scripts/" + methodblueprint["type"] + ".py {method_location} {dataset_location} {output_folder}\n".format(**setting)

# commands_location = "tmp/{settings_name}.txt".format(**locals())
# os.makedirs("../" + os.path.dirname(commands_location), exist_ok=True)
# with open("../" + commands_location, "w") as outfile:
#     outfile.write(commands)
# commands_location = "tmp/{settings_name}.txt".format(**locals())
# os.makedirs(os.path.dirname("../tmp/" + commands_location), exist_ok=True)
# with open("../tmp/" + commands_location, "w") as outfile:
#     outfile.write(commands)
    
# #script_location = generate_batchcode(commands_location, settings_name, len(settings), {"memory":"10G", "numcores":1}, "biclust_comp2")

# # this command can be used on most linux computers to run the different parameter settings in parallel
# print("parallel -j 4 -a " + commands_location)

# Evaluating the method

In [14]:
from modulescomparison import ModevalKnownmodules, ModevalCoverage

Note: If you downloaded the results from zenodo, you don't need to rerun this for "dummy", "agglom", "ica_zscore", "spectral_biclust" and "meanshift"

## By comparing with known modules

Evaluate by comparing with known modules

In [15]:
# create pool of processors
if "pool" in locals().keys():
    pool.close()
pool = mp.Pool(n_jobs)

In [16]:
settings_filtered = [setting for setting in settings if not setting["dataset_name"].startswith("human")] # only evaluate non-human datasets
modeval = ModevalKnownmodules(settings_filtered, baseline = True)

In [17]:
modeval.run(pool)
modeval.save(settings_name)

In [18]:
modeval.load(settings_name)

In [19]:
modeval.scores

,recovery,relevance,F1rr,recall,precision,F1rp,F1rprr,F1rr_permuted,F1rp_permuted,F1rprr_permuted,settingid,knownmodules_name,regnet_name,goldstandard,runningtime
0,0.204448,0.159265,0.179050,0.043916,0.040654,0.042222,0.068331,2.425750,6.073046,3.466772,ecoli_colombos_0,mcl2,ecoli_regulondb,ecoli_regulondb#mcl2,14.320357
1,0.176350,0.115212,0.139371,0.046636,0.023231,0.031013,0.050736,1.894707,3.679719,2.501419,ecoli_colombos_0,minimal,ecoli_regulondb,ecoli_regulondb#minimal,14.320357
2,0.150383,0.100097,0.120192,0.013916,0.019799,0.016344,0.028775,1.599258,2.329778,1.896606,ecoli_colombos_0,ap3,ecoli_regulondb,ecoli_regulondb#ap3,14.320357
3,0.169653,0.128338,0.146131,0.037408,0.026347,0.030918,0.051037,2.188053,4.586968,2.962804,ecoli_colombos_0,tc1,ecoli_regulondb,ecoli_regulondb#tc1,14.320357
4,0.171802,0.135351,0.151414,0.026162,0.024865,0.025497,0.043644,2.378302,4.747873,3.169127,ecoli_colombos_0,mcl3,ecoli_regulondb,ecoli_regulondb#mcl3,14.320357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10675,0.119593,0.219061,0.154719,0.009224,0.118263,0.017114,0.030819,1.371688,0.999170,1.156163,synth_yeast_macisaac_113,ap2,yeast_macisaac,yeast_macisaac#ap2,849.602678
10676,0.110806,0.251119,0.153764,0.021661,0.106059,0.035975,0.058308,2.713513,8.814106,4.149546,synth_yeast_macisaac_113,mcl1,yeast_macisaac,yeast_macisaac#mcl1,849.602678
10677,0.124234,0.276993,0.171534,0.042886,0.078046,0.055354,0.083699,3.002412,7.496605,4.287620,synth_yeast_macisaac_113,tc2,yeast_macisaac,yeast_macisaac#tc2,849.602678
10678,0.076362,0.128968,0.095926,0.004720,0.043453,0.008514,0.015641,1.568116,1.872607,1.706888,synth_yeast_macisaac_113,ap1,yeast_macisaac,yeast_macisaac#ap1,849.602678


## Using the coverage of regulators

In [20]:
# create pool of processors
if "pool" in locals().keys():
    pool.close()
pool = mp.Pool(n_jobs)

In [21]:
settings_filtered = [setting for setting in settings if setting["dataset_name"].startswith("human")] # only evaluate human datasets
modeval = ModevalCoverage(settings_filtered, baseline = True)

In [22]:
modeval.run(pool)
modeval.save(settings_name)

Evaluating a total of 360 settings.


In [23]:
modeval.load(settings_name)

In [24]:
modeval.scores

,aucodds,aucodds_permuted,settingid,goldstandard,runningtime
0,0.133856,5.050977,human_tcga_0,regcircuit,19.557149
1,0.154411,5.826606,human_tcga_18,regcircuit,22.240000
2,0.133856,5.050977,human_tcga_1,regcircuit,19.865595
3,0.133856,5.050977,human_tcga_9,regcircuit,19.884296
4,0.133856,5.050977,human_tcga_2,regcircuit,18.711596
...,...,...,...,...,...
355,0.189069,7.134427,human_seek_gpl5175_109,regcircuit,805.534134
356,0.196258,7.405705,human_seek_gpl5175_117,regcircuit,747.586579
357,0.196258,7.405705,human_seek_gpl5175_110,regcircuit,827.990742
358,0.196258,7.405705,human_seek_gpl5175_118,regcircuit,767.787992


**TODO:** I understand that the previous code generates some files that will be used later to create the final dataframe with scores and the plots.